# Week 4: a transformer hit filter

Same task as last week, but every hit can now attend to every other hit. Compare with the MLP, then look
inside the attention.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
from torch import nn

from exercises.week03.ex02_training_loop import LRConfig, evaluate, train
from minihep.data.dataset import ToyTrackingDataset
from minihep.models.dense import Dense
from minihep.models.encoder import Encoder
from minihep.models.hitfilter import HitFilter
from minihep.models.input import InputNet
from minihep.models.posenc import PositionEncoder
from minihep.models.tasks import HitFilterTask

data_dir = ensure_toy_data(num_train=200 if FAST else 2000, num_val=50 if FAST else 200)
fields = ["x", "y", "z", "r", "eta", "phi"]
cfg = {"inputs": {"hit": fields}, "targets": {"hit": ["on_valid_particle"]}}
train_ds = ToyTrackingDataset(str(data_dir / "train"), **cfg)
val_ds = ToyTrackingDataset(str(data_dir / "val"), **cfg)


def make_filter(encoder: bool, dim=64):
    posenc = PositionEncoder("hit", ["r", "eta", "phi"], dim, sym_fields=["phi"], alpha=10)
    net = InputNet("hit", Dense(len(fields), dim, hidden_layers=[dim]), fields, posenc=posenc)
    enc = Encoder(num_layers=3, dim=dim, attn_kwargs={"num_heads": 4}, norm="LayerNorm") if encoder else None
    task = HitFilterTask("hit_filter", "hit", "on_valid_particle", dim)
    return HitFilter(nn.ModuleList([net]), nn.ModuleList([task]), encoder=enc, input_sort_field="phi")


steps = 300 if FAST else 3000
lr = LRConfig(initial=1e-4, max=1e-3, end=1e-5, pct_start=0.05)
results = {}
for label, use_encoder in (("MLP", False), ("transformer", True)):
    torch.manual_seed(0)
    model = make_filter(use_encoder)
    hist = train(model, train_ds, steps, lr, device=DEVICE)
    results[label] = (model, hist, evaluate(model, val_ds, "hit_filter", device=DEVICE))
    print(label, results[label][2])

In [ ]:
for label, (_, hist, _) in results.items():
    plt.plot(np.convolve(hist.loss, np.ones(50) / 50, mode="valid"), label=label)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("loss (smoothed)")
plt.legend()
plt.show()

## Looking inside: who does a hit attend to?

`F.scaled_dot_product_attention` never hands back the attention weights (that is the point of a fused
kernel), so recompute them for the first layer with the projections and your `naive_attention`.

In [ ]:
from exercises.week04.ex01_attention_from_scratch import naive_attention

model = results["transformer"][0].cpu().eval()
inputs, targets = val_ds[0]
with torch.no_grad():
    x = model.input_nets[0](inputs)
    layer = model.encoder.layers[0]
    attn = layer.attn.fn
    xn = layer.attn.norm(x)
    q, k, v = (attn.separate_heads(t) for t in attn.project_qkv(xn, xn, xn))
    _, weights = naive_attention(q, k, v)  # (1, H, N, N)

hits, _ = val_ds.load_event(0)
query = int(np.flatnonzero(hits.on_valid_particle.to_numpy())[0])
w = weights[0, :, query].mean(0).numpy()
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(hits.x, hits.y, c=w, s=8 + 400 * w / w.max(), cmap="viridis")
same = hits.particle_id == hits.particle_id.iloc[query]
ax.scatter(hits.x[same], hits.y[same], facecolors="none", edgecolors="red", s=80, label="same particle")
ax.scatter(hits.x.iloc[query], hits.y.iloc[query], marker="*", c="red", s=300, label="query hit")
ax.set_aspect("equal")
ax.legend()
ax.set_title("layer-0 attention of one hit, averaged over heads")
plt.show()

## Your turn

1. Remove the positional encoding (`posenc=None`). Does the transformer still beat the MLP? Why might it,
   given the raw x, y, z are still inputs?
2. Try `hybrid_norm=True` in the Encoder and `norm="RMSNorm"`. Any difference at this size?
3. Count parameters of both models. Is the comparison fair?